# Getting Swiss supermarkets using the overpass Web API

## Overpass turbo query to get supermarkets in the city of Zürich

In [2]:
from pandas import json_normalize
import requests
import json
import time
import folium

# Overpass API endpoints to try, in order (some public mirrors are
# unreliable / rate-limited, so we fall back through a list)
urls = [
    'https://overpass.osm.ch/api/interpreter',
    'https://overpass-api.de/api/interpreter',
]

# Overpass turbo query (supermarkets in the city of Zürich)
query = """
        [out:json][timeout:60];
        area["name"="Zürich"]["admin_level"="8"]->.searchArea;
        node["shop"="supermarket"](area.searchArea);
        out;"""

# Web API request with retries (the Overpass servers are often busy)
max_attempts = 6
wait_time = 10  # seconds between attempts
data = None

for attempt in range(1, max_attempts + 1):
    url = urls[(attempt - 1) % len(urls)]
    try:
        r = requests.post(url, data={'data': query}, timeout=90)
        r.raise_for_status()
        data = r.json()['elements']
        print(f'Attempt {attempt} ({url}): success, {len(data)} elements received.')
        break
    except (requests.exceptions.RequestException, ValueError, KeyError) as e:
        print(f'Attempt {attempt} ({url}) failed: {e}')
        if attempt < max_attempts:
            print(f'Waiting {wait_time} seconds before retrying ...')
            time.sleep(wait_time)
            wait_time *= 2  # exponential backoff
        else:
            raise RuntimeError('Overpass API not available after several attempts.')

# Save data to file
with open('supermarkets.json', 'w') as json_file:
    json.dump(data, json_file)

# Store data in data frame
df = json_normalize(data)
df.head(5)

Attempt 1 (https://overpass.osm.ch/api/interpreter): success, 166 elements received.


,type,id,lat,lon,tags.addr:city,tags.addr:country,tags.addr:housenumber,tags.addr:postcode,tags.addr:street,tags.brand,...,tags.stroller,tags.toilets,tags.operator:wikidata,tags.membership,tags.type,tags.payment:notes,tags.cuisine,tags.contact:instagram,tags.facebook,tags.fair_trade
0,node,39947904,47.376732,8.542161,Zürich,CH,1,8001,Bahnhofbrücke,Coop,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,48932835,47.375020,8.522895,Zürich,NaN,7,8004,Wengistrasse,Migros,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,75749133,47.340967,8.530601,Zürich,CH,81,8038,Albisstrasse,ALDI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,79977755,47.340070,8.530546,Zürich,CH,15,8038,Alte Kalchbühlstrasse,Coop,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,83330862,47.344749,8.529981,Zürich,CH,3,8038,Etzelstrasse,Migros,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Plot supermarkets on map

In [3]:
# Subset of supermarkets by brand
locations = df[["lat", "lon", "tags.brand", "tags.shop"]].loc[df["tags.brand"].isin(['Coop', 'Migros'])]
print(locations.head(5))

# Subsets
locations_coop = df[["lat", "lon", "tags.brand",
                     "tags.shop"]].loc[df["tags.brand"].isin(['Coop'])]
locations_migros = df[["lat", "lon", "tags.brand",
                       "tags.shop"]].loc[df["tags.brand"].isin(['Migros'])]

# Marker symbols
url_01 = 'https://raw.githubusercontent.com/pointhi/leaflet-color-markers/master/img/marker-icon-green.png'
url_02 = 'https://raw.githubusercontent.com/pointhi/leaflet-color-markers/master/img/marker-icon-orange.png'

# Create map
map = folium.Map(location=[locations.lat.mean(),
                           locations.lon.mean()],
                 zoom_start=12,
                 control_scale=True)

# Add lat/lon of Coop supermarkets
for i in range(0, len(locations_coop)):
    folium.Marker(location=(locations_coop.iloc[i]['lat'],
                            locations_coop.iloc[i]['lon']),
                  popup=locations_coop.iloc[i]['tags.brand'],
                  icon=folium.features.CustomIcon(url_01, icon_size=(16, 28))).add_to(map)
    
# Add lat/lon of Migros supermarkets
for i in range(0, len(locations_migros)):
    folium.Marker(location=(locations_migros.iloc[i]['lat'],
                            locations_migros.iloc[i]['lon']),
                  popup=locations_migros.iloc[i]['tags.brand'],
                  icon=folium.features.CustomIcon(url_02, icon_size=(16, 28))).add_to(map)

# Plot map
map

         lat       lon tags.brand    tags.shop
0  47.376732  8.542161       Coop  supermarket
1  47.375020  8.522895     Migros  supermarket
3  47.340070  8.530546       Coop  supermarket
4  47.344749  8.529981     Migros  supermarket
5  47.375255  8.536107     Migros  supermarket


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')